# MisEdit_Audit.ipynb
**Paper:** MisEdit — Misconception Editing in Sub-3B Language Models

**Sections:**
- Section 1: MythBench dataset audit + 5-cluster assignment
- Section 2: EasyEdit setup + ROME compatibility (Qwen2.5-1.5B, TinyLlama-1.1B)
- Section 3: Evaluation functions (MC scoring v2)
- Section 4: Diagnostic — 5 misconceptions, pre/post generation + MC

---
## SECTION 1: MythBench Audit

In [1]:
# Cell 1.1 — Load MythBench
import json, os

JSON_PATH = "/kaggle/input/datasets/kevinsam77/mythbench-dataset/MythBench_v10.json"

with open(JSON_PATH) as f:
    data = json.load(f)

misconceptions = data["misconceptions"]
controls = data["controls"]

print("=== Load successful ===")
print(f"Version: {data.get('version')}")
print(f"Note: {data.get('note')}")
print(f"Misconceptions: {len(misconceptions)}")
print(f"Controls: {len(controls)}")
print(f"Fields: {list(misconceptions[0].keys())}")

=== Load successful ===
Version: 1.0
Note: MythBench v1.0 — 48 empirically validated misconception items + 48 matched controls. All misconceptions validated via Gemma-2B attraction screening (>60% threshold). Options shuffled.
Misconceptions: 48
Controls: 48
Fields: ['id', 'misconception_label', 'category', 'question', 'options', 'correct', 'misconception', 'source']


In [2]:
# Cell 1.2 — 5-cluster assignment
CLUSTERS = {
    "Health_HumanBiology": [
        "antibiotics_kill_viruses", "sugar_causes_hyperactivity_children",
        "tongue_taste_zones", "hair_nails_grow_after_death",
        "carrots_improve_night_vision", "cold_weather_causes_colds",
        "cracking_knuckles_causes_arthritis", "humans_lose_most_heat_through_head",
        "humans_swallow_spiders_in_sleep", "coffee_dehydrates_you",
        "left_brain_right_brain_dominance", "shaving_makes_hair_grow_thicker",
        "alcohol_warms_you_up"
    ],
    "Animals_NaturalScience": [
        "bats_are_blind", "camels_store_water_in_humps",
        "bulls_enraged_by_red_color", "chameleons_camouflage_surroundings",
        "ostriches_bury_head_in_sand"
    ],
    "Physics_Earth_Space": [
        "great_wall_visible_from_space", "seasons_caused_by_distance_from_sun",
        "toilet_flush_coriolis_hemispheres", "glass_is_slow_liquid",
        "Mount_Everest_coldest_place_earth", "Mt_everest_highest_point_atmosphere",
        "water_drains_opposite_hemispheres", "diamond_hardest_material_earth"
    ],
    "History_Civilization": [
        "vikings_wore_horned_helmets", "marie_antoinette_let_them_eat_cake",
        "columbus_proved_earth_round", "cleopatra_was_egyptian",
        "nero_fiddled_while_rome_burned", "pilgrims_wore_black_and_white",
        "roman_vomitoriums_for_vomiting", "isaac_newton_apple_fell_on_head",
        "witch_trials_burned_at_stake_in_salem", "great_wall_built_to_keep_mongols_out",
        "ben_franklin_discovered_electricity", "lincoln_born_in_log_cabin_poverty",
        "cold_war_never_had_direct_combat", "signing_declaration_independence_july_4",
        "walt_disney_drew_mickey_mouse"
    ],
    "Geography_CulturalOrigins": [
        "fortune_cookies_chinese_origin", "french_fries_invented_in_france",
        "chinese_invented_pasta_marco_polo", "america_named_after_amerigo_vespucci",
        "alaska_is_northernmost_us_state", "pacific_ocean_named_for_being_calm",
        "mount_olympus_home_of_gods_in_clouds"
    ]
}

label_to_cluster = {}
for cluster, labels in CLUSTERS.items():
    for label in labels:
        label_to_cluster[label] = cluster

print("=== Cluster Assignment ===")
for cluster, labels in CLUSTERS.items():
    count = sum(1 for label in labels if any(m["misconception_label"] == label for m in misconceptions))
    print(f"  {cluster}: {count} items")

unassigned = [m["misconception_label"] for m in misconceptions if m["misconception_label"] not in label_to_cluster]
print(f"\nUnassigned: {len(unassigned)}")
total = sum(len(v) for v in CLUSTERS.values())
print(f"Total assigned: {total} / {len(misconceptions)}")
print(f"Controls available: {len(controls)}")

=== Cluster Assignment ===
  Health_HumanBiology: 13 items
  Animals_NaturalScience: 5 items
  Physics_Earth_Space: 8 items
  History_Civilization: 15 items
  Geography_CulturalOrigins: 7 items

Unassigned: 0
Total assigned: 48 / 48
Controls available: 48


---
## SECTION 2: EasyEdit Setup + ROME Compatibility

In [5]:
# Cell 2.1 — EasyEdit setup (Kaggle-safe, consolidated)
import sys

!git clone https://github.com/zjunlp/EasyEdit.git /kaggle/working/EasyEdit 2>/dev/null || echo "Already cloned"
sys.path.insert(0, '/kaggle/working/EasyEdit')

!pip install einops hydra-core higher omegaconf sentence-transformers peft \
             sentencepiece rouge av qwen-vl-utils iopath fairscale zhipuai -q

# Patch wikipedia dataset loader (deprecated in newer datasets library)
memit_file = "/kaggle/working/EasyEdit/easyeditor/models/rome/layer_stats.py"
with open(memit_file) as f:
    content = f.read()
content = content.replace(
    'dict(wikitext="wikitext-103-raw-v1", wikipedia="20200501.en")[ds_name]',
    'dict(wikitext="wikitext-103-raw-v1", wikipedia="wikitext-103-raw-v1")[ds_name]'
)
with open(memit_file, "w") as f:
    f.write(content)
print("layer_stats.py patched")

try:
    from easyeditor import BaseEditor
    print("EasyEdit: SUCCESS")
except ImportError as e:
    print(f"EasyEdit: FAILED — {e}")

import torch, transformers
print(f"Torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"Transformers: {transformers.__version__}")

Already cloned
layer_stats.py patched
EasyEdit: SUCCESS
Torch: 2.10.0+cu128 | CUDA: True
Transformers: 5.0.0


In [6]:
# Cell 2.2 — Write hparams yamls for TinyLlama and Qwen2.5-1.5B
import yaml, shutil

hparams_path = "/kaggle/working/EasyEdit/hparams"

# TinyLlama — copy from llama3.2-3b, fix layer indices for 22-layer model
for method in ["ROME", "MEMIT"]:
    src = f"{hparams_path}/{method}/llama3.2-3b.yaml"
    dst = f"{hparams_path}/{method}/LlamaForCausalLM.yaml"
    shutil.copy(src, dst)
    with open(dst) as f:
        cfg = yaml.safe_load(f)
    cfg['model_name'] = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
    cfg['device'] = 0
    cfg['v_loss_layer'] = 21
    cfg['mom2_adjustment'] = False
    cfg['v_num_grad_steps'] = 10
    if 'layers' in cfg:
        cfg['layers'] = [i for i in cfg['layers'] if i < 22]
    with open(dst, "w") as f:
        yaml.dump(cfg, f)

# Qwen2.5-1.5B — copy from qwen2.5-7b, fix dims
for method in ["ROME", "MEMIT"]:
    src = f"{hparams_path}/{method}/qwen2.5-7b.yaml"
    dst = f"{hparams_path}/{method}/Qwen2ForCausalLM.yaml"
    shutil.copy(src, dst)
    with open(dst) as f:
        cfg = yaml.safe_load(f)
    cfg['model_name'] = 'Qwen/Qwen2.5-1.5B-Instruct'
    cfg['device'] = 0
    cfg['mom2_adjustment'] = False
    cfg['v_num_grad_steps'] = 10
    cfg['rewrite_module_tmp'] = 'model.layers.{}.mlp.down_proj'
    if method == "MEMIT":
        cfg['lm_head_module'] = 'model.embed_tokens'
    with open(dst, "w") as f:
        yaml.dump(cfg, f)

print("Hparams written for TinyLlama + Qwen2.5-1.5B (ROME + MEMIT)")

Hparams written for TinyLlama + Qwen2.5-1.5B (ROME + MEMIT)


---
## SECTION 3: Evaluation Functions

In [7]:
# Cell 3.1 — MC scoring v2 (option text probability, aligns with ROME edits)
import torch
import torch.nn.functional as F

def score_mc_belief_v2(model, tokenizer, question, options, correct_key, misconception_key):
    """
    Score each MC option by average log-probability of option text given question.
    Aligns with ROME's editing objective (text generation probability).
    """
    scores = {}
    question_prompt = f"{question}\nAnswer:"
    question_ids = tokenizer.encode(question_prompt, return_tensors="pt").to(model.device)

    for key, option_text in options.items():
        prompt = f"{question}\nAnswer: {option_text}"
        full_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
        option_len = full_ids.shape[1] - question_ids.shape[1]
        if option_len <= 0:
            scores[key] = -999.0
            continue
        with torch.no_grad():
            outputs = model(full_ids)
            logits = outputs.logits[0]
        log_probs = F.log_softmax(logits, dim=-1)
        option_start = question_ids.shape[1] - 1
        total = sum(
            log_probs[option_start + i, full_ids[0, question_ids.shape[1] + i].item()].item()
            for i in range(option_len)
        )
        scores[key] = total / option_len

    best_key = max(scores, key=scores.get)
    return {
        "correct_prob": scores.get(correct_key, -999.0),
        "misconception_prob": scores.get(misconception_key, -999.0),
        "predicted_key": best_key,
        "is_correct": best_key == correct_key,
        "is_misconception": best_key == misconception_key,
        "raw_scores": scores
    }


def validate_misconception_present(model, tokenizer, item, threshold=2):
    """
    3-prompt majority vote to confirm misconception is present pre-edit.
    Returns (present: bool, vote_count: int)
    """
    question = item["question"]
    options = item["options"]
    correct_key = item["correct"]
    misconception_key = item["misconception"]

    votes = 0
    # Prompt A: direct
    r = score_mc_belief_v2(model, tokenizer, question, options, correct_key, misconception_key)
    if r["is_misconception"]: votes += 1
    # Prompt B: True/False
    tf_q = f"True or False: {options[misconception_key]}"
    tf_opts = {"A": "True", "B": "False"}
    r2 = score_mc_belief_v2(model, tokenizer, tf_q, tf_opts, "B", "A")
    if r2["is_misconception"]: votes += 1
    # Prompt C: rephrase
    r3 = score_mc_belief_v2(model, tokenizer, f"Which is correct?\n{question}", options, correct_key, misconception_key)
    if r3["is_misconception"]: votes += 1

    return votes >= threshold, votes

print("Evaluation functions defined.")

Evaluation functions defined.


---
## SECTION 4: Diagnostic — 5 Misconceptions, Pre/Post ROME

In [8]:
# Cell 4.1 — Load 5 diagnostic misconceptions (one per cluster)
DIAG_LABELS = [
    "great_wall_visible_from_space",       # Physics_Earth_Space
    "bats_are_blind",                      # Animals_NaturalScience
    "antibiotics_kill_viruses",            # Health_HumanBiology
    "vikings_wore_horned_helmets",         # History_Civilization
    "seasons_caused_by_distance_from_sun"  # Physics_Earth_Space (control)
]

diag_items = [m for m in misconceptions if m["misconception_label"] in DIAG_LABELS]
print(f"Loaded {len(diag_items)} diagnostic items:")
for item in diag_items:
    print(f"  [{item['misconception_label']}] correct={item['correct']} misconception={item['misconception']}")

Loaded 5 diagnostic items:
  [great_wall_visible_from_space] correct=B misconception=C
  [vikings_wore_horned_helmets] correct=A misconception=B
  [bats_are_blind] correct=A misconception=B
  [seasons_caused_by_distance_from_sun] correct=B misconception=A
  [antibiotics_kill_viruses] correct=D misconception=B


In [9]:
# Cell 4.2 — Diagnostic function
from transformers import AutoTokenizer, AutoModelForCausalLM
from easyeditor import BaseEditor, ROMEHyperParams
import warnings
warnings.filterwarnings("ignore")

def run_diagnostic(model, tokenizer, editor, item):
    question = item["question"]
    options = item["options"]
    correct_key = item["correct"]
    misconception_key = item["misconception"]

    # Extract subject from question (rough heuristic)
    # Extract subject — must be a substring of the question
    # Use first noun phrase that appears in the question
    words = question.split()
    # Try progressively shorter spans from the start after question words
    subject = None
    skip = {"Is", "Did", "Do", "Does", "Can", "Are", "Was", "Were", "Have", "Has"}
    start = 1 if words[0] in skip else 0
    for length in range(min(6, len(words)-start), 0, -1):
        candidate = " ".join(words[start:start+length])
        if candidate in question:
            subject = candidate
            break
    if subject is None:
        subject = words[start]

    # PRE-EDIT: free generation
    inputs = tokenizer(question, return_tensors="pt").to(model.device)
    with torch.no_grad():
        gen_ids = model.generate(**inputs, max_new_tokens=30, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
    pre_gen = tokenizer.decode(gen_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    # PRE-EDIT: MC scoring
    pre_mc = score_mc_belief_v2(model, tokenizer, question, options, correct_key, misconception_key)

    # APPLY ROME
    _, edited_model, _ = editor.edit(
        prompts=[question],
        rephrase_prompts=[question],
        target_new=[options[correct_key]],
        subject=[subject],
        keep_original_weight=True
    )

    # POST-EDIT: free generation
    inputs2 = tokenizer(question, return_tensors="pt").to(edited_model.device)
    with torch.no_grad():
        gen_ids2 = edited_model.generate(**inputs2, max_new_tokens=30, do_sample=False,
                                          pad_token_id=tokenizer.eos_token_id)
    post_gen = tokenizer.decode(gen_ids2[0][inputs2["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    # POST-EDIT: MC scoring
    post_mc = score_mc_belief_v2(edited_model, tokenizer, question, options, correct_key, misconception_key)

    del edited_model
    torch.cuda.empty_cache()

    return {
        "label": item["misconception_label"],
        "pre_gen": pre_gen,
        "post_gen": post_gen,
        "pre_mc_predicted": pre_mc["predicted_key"],
        "pre_mc_correct_score": pre_mc["correct_prob"],
        "pre_mc_misc_score": pre_mc["misconception_prob"],
        "post_mc_predicted": post_mc["predicted_key"],
        "post_mc_correct_score": post_mc["correct_prob"],
        "post_mc_misc_score": post_mc["misconception_prob"],
        "gen_changed": pre_gen[:60] != post_gen[:60],
        "mc_predicted_changed": pre_mc["predicted_key"] != post_mc["predicted_key"],
        "pre_held_misconception": pre_mc["is_misconception"],
        "post_held_misconception": post_mc["is_misconception"],
    }

print("Diagnostic function defined.")

Diagnostic function defined.


In [10]:
# Cell 4.3 — Skip Qwen (OOM), run diagnostic on TinyLlama-1.1B + ROME only
import torch, gc
import gc, os

torch.cuda.empty_cache()
gc.collect()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
os.chdir('/kaggle/working/EasyEdit')

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
HPARAMS  = "hparams/ROME/LlamaForCausalLM.yaml"

tokenizer_q = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer_q.pad_token is None:
    tokenizer_q.pad_token = tokenizer_q.eos_token

model_q = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="cuda:0", trust_remote_code=True
)

torch.cuda.empty_cache()
gc.collect()

hparams_q = ROMEHyperParams.from_hparams(HPARAMS)
hparams_q.model_parallel = False
hparams_q.device = 0
editor_q = BaseEditor.from_hparams(hparams_q)

results_tinyllama = []
for item in diag_items:
    print(f"\n--- {item['misconception_label']} ---")
    r = run_diagnostic(model_q, tokenizer_q, editor_q, item)
    results_tinyllama.append(r)
    print(f"  PRE  gen:  {r['pre_gen'][:80]}")
    print(f"  POST gen:  {r['post_gen'][:80]}")
    print(f"  PRE  MC:   predicted={r['pre_mc_predicted']}  misc_score={r['pre_mc_misc_score']:.3f}")
    print(f"  POST MC:   predicted={r['post_mc_predicted']}  misc_score={r['post_mc_misc_score']:.3f}")
    print(f"  gen_changed={r['gen_changed']}  mc_changed={r['mc_predicted_changed']}")
    torch.cuda.empty_cache()
    gc.collect()

del model_q
torch.cuda.empty_cache()
gc.collect()
print("TinyLlama diagnostic complete.")

06/07/2026 17:41:07 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06/07/2026 17:41:07 - WARNING - huggingface_hub.utils._http -   Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
06/07/2026 17:41:07 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"


GPU free: 15.5 GB


06/07/2026 17:41:07 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
06/07/2026 17:41:07 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/tokenizer_config.json "HTTP/1.1 200 OK"
06/07/2026 17:41:07 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
06/07/2026 17:41:07 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/tokenizer_config.json "HTTP/1.1 200 OK"
06/07/2026 17:41:07 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/tree/main/additional_chat_templates?recursive=false&expand=f

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

06/07/2026 17:41:09 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
06/07/2026 17:41:09 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/generation_config.json "HTTP/1.1 200 OK"
06/07/2026 17:41:09 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"
2026-06-07 17:41:09,949 - easyeditor.editors.editor - INFO - Instantiating model
06/07/2026 17:41:09 - INFO - easyeditor.editors.editor -   Instantiating model
06/07/2026 17:41:09 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06/07/2026 17:41:10 - INFO - httpx -   HTTP Request: HEAD https://huggingf

...

  0%|          | 0/1 [00:00<?, ?it/s]

Executing ROME algorithm for the update: [Can antibiotics be used to treat viral infections like the common cold or flu?] -> [ No, antibiotics only work against bacterial infections and have no effect on viruses]
Computing left vector (u)...
Selected u projection object antibiotics be used to treat viral
Left vector shape: torch.Size([5632])
Computing right vector (v)
Lookup index found: 11 | Sentence: Can antibiotics be used to treat viral infections like the common cold or flu? No, antibiotics only work against bacterial infections and have no effect on vir | Token: al
Rewrite layer is 5
Tying optimization objective to 21
Recording initial value of v*
loss 1.381 = 1.381 + 0.0 + 0.0 avg prob of [ No, antibiotics only work against bacterial infections and have no effect on viruses] 0.25257351994514465
loss 1.381 = 1.311 + 0.067 + 0.003 avg prob of [ No, antibiotics only work against bacterial infections and have no effect on viruses] 0.2708374261856079
loss 1.322 = 1.307 + 0.012 + 0.00

2026-06-07 17:41:35,264 - easyeditor.editors.editor - INFO - 0 editing: Can antibiotics be used to treat viral infections like the common cold or flu? -> No, antibiotics only work against bacterial infections and have no effect on viruses  

 {'pre': {'rewrite_acc': [np.float64(0.7727272727272727)], 'portability': {}, 'rephrase_acc': [np.float64(0.7727272727272727)]}, 'case_id': 0, 'requested_rewrite': {'prompt': 'Can antibiotics be used to treat viral infections like the common cold or flu?', 'target_new': 'No, antibiotics only work against bacterial infections and have no effect on viruses', 'ground_truth': '<|endoftext|>', 'portability': {}, 'locality': {}, 'subject': 'antibiotics be used to treat viral', 'rephrase_prompt': 'Can antibiotics be used to treat viral infections like the common cold or flu?'}, 'post': {'rewrite_acc': [np.float64(0.7272727272727273)], 'locality': {}, 'portability': {}, 'rephrase_acc': [np.float64(0.7272727272727273)]}}
06/07/2026 17:41:35 - INFO - easyedi

loss 1.094 = 1.083 + 0.008 + 0.003 avg prob of [ No, antibiotics only work against bacterial infections and have no effect on viruses] 0.3403584659099579
Delta norm: 5.35546875
Change in target norm: 1.3388671875 to 5.50390625 => 4.1640625
Division Factor: 1.52734375
Right vector norm: 3.505859375
Right vector shape: torch.Size([2048])
Deltas successfully computed for ['model.layers.5.mlp.down_proj.weight']
New weights successfully inserted into ['model.layers.5.mlp.down_proj.weight']
Metrics Summary:  {'pre': {'rewrite_acc': np.float64(0.7727272727272727), 'rephrase_acc': np.float64(0.7727272727272727)}, 'post': {'rewrite_acc': np.float64(0.7272727272727273), 'rephrase_acc': np.float64(0.7272727272727273)}}


  PRE  gen:  
  POST gen:  
  PRE  MC:   predicted=D  misc_score=-1.123
  POST MC:   predicted=D  misc_score=-1.123
  gen_changed=False  mc_changed=False
TinyLlama diagnostic complete.


In [ ]:
# Cell 4.4 — Run diagnostic on TinyLlama-1.1B + ROME
MODEL_ID_T = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
HPARAMS_T  = "hparams/ROME/LlamaForCausalLM.yaml"

tokenizer_t = AutoTokenizer.from_pretrained(MODEL_ID_T, trust_remote_code=True)
if tokenizer_t.pad_token is None:
    tokenizer_t.pad_token = tokenizer_t.eos_token

model_t = AutoModelForCausalLM.from_pretrained(
    MODEL_ID_T, dtype=torch.float16, device_map="cuda:0", trust_remote_code=True
)

hparams_t = ROMEHyperParams.from_hparams(HPARAMS_T)
editor_t  = BaseEditor.from_hparams(hparams_t)

results_tinyllama = []
for item in diag_items:
    print(f"\n--- {item['misconception_label']} ---")
    r = run_diagnostic(model_t, tokenizer_t, editor_t, item)
    results_tinyllama.append(r)
    print(f"  PRE  gen:  {r['pre_gen'][:80]}")
    print(f"  POST gen:  {r['post_gen'][:80]}")
    print(f"  PRE  MC:   predicted={r['pre_mc_predicted']}  misc_score={r['pre_mc_misc_score']:.3f}")
    print(f"  POST MC:   predicted={r['post_mc_predicted']}  misc_score={r['post_mc_misc_score']:.3f}")
    print(f"  gen_changed={r['gen_changed']}  mc_changed={r['mc_predicted_changed']}")

del model_t
torch.cuda.empty_cache()

In [ ]:
# Cell 4.5 — Summary table
import pandas as pd

def make_summary(results, model_name):
    rows = []
    for r in results:
        rows.append({
            "Model": model_name,
            "Misconception": r["label"],
            "Pre held misc": r["pre_held_misconception"],
            "Post held misc": r["post_held_misconception"],
            "Gen changed": r["gen_changed"],
            "MC changed": r["mc_predicted_changed"],
            "Pre misc score": round(r["pre_mc_misc_score"], 3),
            "Post misc score": round(r["post_mc_misc_score"], 3),
        })
    return pd.DataFrame(rows)

summary = make_summary(results_tinyllama, "TinyLlama-1.1B")


print("=== Diagnostic Summary ===")
print(summary.to_string(index=False))

print("\n=== Key Questions ===")
print(f"Cases where Gen changed but MC unchanged: {((summary['Gen changed']) & (~summary['MC changed'])).sum()}")
print(f"Cases where both changed: {(summary['Gen changed'] & summary['MC changed']).sum()}")
print(f"Cases where neither changed: {(~summary['Gen changed'] & ~summary['MC changed']).sum()}")

summary.to_csv("/kaggle/working/MisEdit_Diagnostic_Results.csv", index=False)
print("\nSaved: /kaggle/working/MisEdit_Diagnostic_Results.csv")

In [11]:
# Cell 4.6 — Option A: Short target reformulation test
# Test whether shorter/different targets change MC or generation
import gc, os
os.chdir('/kaggle/working/EasyEdit')

# 3 test cases with different target formulations
SHORT_TESTS = [
    {
        "item": next(m for m in misconceptions if m["misconception_label"] == "great_wall_visible_from_space"),
        "targets": [
            "False",
            "No",
            "The Great Wall is not visible from space."
        ]
    },
    {
        "item": next(m for m in misconceptions if m["misconception_label"] == "antibiotics_kill_viruses"),
        "targets": [
            "False",
            "No",
            "Antibiotics do not work against viruses."
        ]
    },
    {
        "item": next(m for m in misconceptions if m["misconception_label"] == "vikings_wore_horned_helmets"),
        "targets": [
            "False",
            "No",
            "Vikings did not wear horned helmets."
        ]
    }
]

def test_short_target(model, tokenizer, editor, item, target):
    question = item["question"]
    options = item["options"]
    correct_key = item["correct"]
    misconception_key = item["misconception"]

    # Pre-edit MC
    pre_mc = score_mc_belief_v2(model, tokenizer, question, options, correct_key, misconception_key)

    # Apply ROME with short target
    words = question.split()
    skip = {"Is","Did","Do","Does","Can","Are","Was","Were","Have","Has"}
    start = 1 if words[0] in skip else 0
    subject = None
    for length in range(min(6, len(words)-start), 0, -1):
        candidate = " ".join(words[start:start+length])
        if candidate in question:
            subject = candidate
            break
    if subject is None:
        subject = words[start]

    _, edited_model, _ = editor.edit(
        prompts=[question],
        rephrase_prompts=[question],
        target_new=[target],
        subject=[subject],
        keep_original_weight=True
    )

    # Post-edit MC
    post_mc = score_mc_belief_v2(edited_model, tokenizer, question, options, correct_key, misconception_key)

    # Post-edit generation
    inputs = tokenizer(question, return_tensors="pt").to(edited_model.device)
    with torch.no_grad():
        gen_ids = edited_model.generate(**inputs, max_new_tokens=20, do_sample=False,
                                         pad_token_id=tokenizer.eos_token_id)
    post_gen = tokenizer.decode(gen_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    del edited_model
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "pre_mc_predicted": pre_mc["predicted_key"],
        "post_mc_predicted": post_mc["predicted_key"],
        "pre_misc_score": pre_mc["misconception_prob"],
        "post_misc_score": post_mc["misconception_prob"],
        "mc_changed": pre_mc["predicted_key"] != post_mc["predicted_key"],
        "post_gen": post_gen
    }

# Run — model still loaded from Cell 4.3
# Need to reload TinyLlama since it was deleted
tokenizer_a = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0", trust_remote_code=True)
if tokenizer_a.pad_token is None:
    tokenizer_a.pad_token = tokenizer_a.eos_token
model_a = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype=torch.float16, device_map="cuda:0", trust_remote_code=True
)
hparams_a = ROMEHyperParams.from_hparams("hparams/ROME/LlamaForCausalLM.yaml")
hparams_a.model_parallel = False
hparams_a.device = 0
editor_a = BaseEditor.from_hparams(hparams_a)

print("=== Option A: Short Target Test ===\n")
for test in SHORT_TESTS:
    item = test["item"]
    print(f"[{item['misconception_label']}]")
    for target in test["targets"]:
        r = test_short_target(model_a, tokenizer_a, editor_a, item, target)
        print(f"  target='{target[:40]}'")
        print(f"    MC: {r['pre_mc_predicted']} → {r['post_mc_predicted']} | misc_score: {r['pre_misc_score']:.3f} → {r['post_misc_score']:.3f} | changed={r['mc_changed']}")
        print(f"    gen: {r['post_gen'][:70]}")
    print()

del model_a
torch.cuda.empty_cache()
gc.collect()

06/07/2026 17:46:19 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06/07/2026 17:46:19 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
06/07/2026 17:46:19 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
06/07/2026 17:46:19 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/tokenizer_config.json "HTTP/1.1 200 OK"
06/07/2026 17:46:19 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
06/07/2026 17:46:19 -

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

06/07/2026 17:46:21 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
06/07/2026 17:46:21 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/generation_config.json "HTTP/1.1 200 OK"
06/07/2026 17:46:21 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"
2026-06-07 17:46:21,292 - easyeditor.editors.editor - INFO - Instantiating model
2026-06-07 17:46:21,292 - easyeditor.editors.editor - INFO - Instantiating model
06/07/2026 17:46:21 - INFO - easyeditor.editors.editor -   Instantiating model
06/07/2026 17:46:21 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Red

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

06/07/2026 17:46:22 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
06/07/2026 17:46:22 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/generation_config.json "HTTP/1.1 200 OK"
06/07/2026 17:46:22 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06/07/2026 17:46:22 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
06/07/2026 17:46:22 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
06/07/2026 17:46:22

...

Executing ROME algorithm for the update: [Did Viking warriors typically wear horned helmets in battle?] -> [ Vikings did not wear horned helmets.]
Computing left vector (u)...
Selected u projection object Viking warriors typically wear horned helmets
Left vector shape: torch.Size([5632])
Computing right vector (v)
Lookup index found: 12 | Sentence: Did Viking warriors typically wear horned helmets in battle? Vikings did not wear horned helmets | Token: ets
Rewrite layer is 5
Tying optimization objective to 21
Recording initial value of v*
loss 1.904 = 1.904 + 0.0 + 0.0 avg prob of [ Vikings did not wear horned helmets.] 0.15274450182914734
loss 1.618 = 1.594 + 0.02 + 0.004 avg prob of [ Vikings did not wear horned helmets.] 0.20688055455684662
loss 1.279 = 1.125 + 0.15 + 0.004 avg prob of [ Vikings did not wear horned helmets.] 0.3263174295425415
loss 1.382 = 1.353 + 0.025 + 0.004 avg prob of [ Vikings did not wear horned helmets.] 0.26179060339927673
loss 0.848 = 0.793 + 0.051 + 0.004

2026-06-07 17:46:54,626 - easyeditor.editors.editor - INFO - 0 editing: Did Viking warriors typically wear horned helmets in battle? -> Vikings did not wear horned helmets.  

 {'pre': {'rewrite_acc': [np.float64(0.6363636363636364)], 'portability': {}, 'rephrase_acc': [np.float64(0.6363636363636364)]}, 'case_id': 0, 'requested_rewrite': {'prompt': 'Did Viking warriors typically wear horned helmets in battle?', 'target_new': 'Vikings did not wear horned helmets.', 'ground_truth': '<|endoftext|>', 'portability': {}, 'locality': {}, 'subject': 'Viking warriors typically wear horned helmets', 'rephrase_prompt': 'Did Viking warriors typically wear horned helmets in battle?'}, 'post': {'rewrite_acc': [np.float64(1.0)], 'locality': {}, 'portability': {}, 'rephrase_acc': [np.float64(1.0)]}}
2026-06-07 17:46:54,626 - easyeditor.editors.editor - INFO - 0 editing: Did Viking warriors typically wear horned helmets in battle? -> Vikings did not wear horned helmets.  

 {'pre': {'rewrite_acc': [np.

loss 0.509 = 0.351 + 0.154 + 0.004 avg prob of [ Vikings did not wear horned helmets.] 0.7049300670623779
Delta norm: 4.546875
Change in target norm: 1.13671875 to 4.6484375 => 3.51171875
Division Factor: 1.30078125
Right vector norm: 3.49609375
Right vector shape: torch.Size([2048])
Deltas successfully computed for ['model.layers.5.mlp.down_proj.weight']
New weights successfully inserted into ['model.layers.5.mlp.down_proj.weight']
Metrics Summary:  {'pre': {'rewrite_acc': np.float64(0.6363636363636364), 'rephrase_acc': np.float64(0.6363636363636364)}, 'post': {'rewrite_acc': np.float64(1.0), 'rephrase_acc': np.float64(1.0)}}


  target='Vikings did not wear horned helmets.'
    MC: B → B | misc_score: -1.461 → -1.461 | changed=False
    gen: 



0

In [12]:
# MisEdit — Qwen2.5-1.5B Replication Diagnostic
# Self-contained: installs, patches, runs, prints results

import subprocess, sys, os, json, torch, gc
import torch.nn.functional as F

# ── 1. Install EasyEdit ──────────────────────────────────────────────────────
subprocess.run(["git", "clone", "https://github.com/zjunlp/EasyEdit.git",
                "/kaggle/working/EasyEdit"], capture_output=True)
sys.path.insert(0, '/kaggle/working/EasyEdit')
os.chdir('/kaggle/working/EasyEdit')

subprocess.run([sys.executable, "-m", "pip", "install",
                "einops", "hydra-core", "higher", "omegaconf",
                "sentence-transformers", "peft", "sentencepiece",
                "rouge", "av", "qwen-vl-utils", "iopath",
                "fairscale", "zhipuai", "-q"], capture_output=True)

# Patch deprecated wikipedia loader
f = "/kaggle/working/EasyEdit/easyeditor/models/rome/layer_stats.py"
txt = open(f).read().replace(
    'dict(wikitext="wikitext-103-raw-v1", wikipedia="20200501.en")[ds_name]',
    'dict(wikitext="wikitext-103-raw-v1", wikipedia="wikitext-103-raw-v1")[ds_name]'
)
open(f, "w").write(txt)

from easyeditor import BaseEditor, ROMEHyperParams
from transformers import AutoTokenizer, AutoModelForCausalLM
import yaml, shutil
print("EasyEdit ready.")

# ── 2. Write Qwen hparams yaml ───────────────────────────────────────────────
hparams_path = "/kaggle/working/EasyEdit/hparams"
src = f"{hparams_path}/ROME/qwen2.5-7b.yaml"
dst = f"{hparams_path}/ROME/Qwen2ForCausalLM.yaml"
shutil.copy(src, dst)
with open(dst) as f_:
    cfg = yaml.safe_load(f_)
cfg['model_name'] = 'Qwen/Qwen2.5-1.5B-Instruct'
cfg['device'] = 0
cfg['mom2_adjustment'] = False
cfg['v_num_grad_steps'] = 10
cfg['rewrite_module_tmp'] = 'model.layers.{}.mlp.down_proj'
with open(dst, "w") as f_:
    yaml.dump(cfg, f_)
print("Hparams written.")

# ── 3. Evaluation function ───────────────────────────────────────────────────
def score_mc(model, tokenizer, question, options, correct_key, misc_key):
    q_prompt = f"{question}\nAnswer:"
    q_ids = tokenizer.encode(q_prompt, return_tensors="pt").to(model.device)
    scores = {}
    for key, text in options.items():
        full_ids = tokenizer.encode(f"{question}\nAnswer: {text}",
                                     return_tensors="pt").to(model.device)
        opt_len = full_ids.shape[1] - q_ids.shape[1]
        if opt_len <= 0:
            scores[key] = -999.0; continue
        with torch.no_grad():
            logits = model(full_ids).logits[0]
        lp = F.log_softmax(logits, dim=-1)
        start = q_ids.shape[1] - 1
        scores[key] = sum(lp[start+i, full_ids[0, q_ids.shape[1]+i].item()].item()
                          for i in range(opt_len)) / opt_len
    best = max(scores, key=scores.get)
    return {"predicted": best,
            "correct_score": scores[correct_key],
            "misc_score": scores[misc_key],
            "is_misc": best == misc_key}

# ── 4. Load data + model ─────────────────────────────────────────────────────
DATA = "/kaggle/input/datasets/kevinsam77/mythbench-dataset/MythBench_v10.json"
with open(DATA) as f_:
    data = json.load(f_)
misconceptions = data["misconceptions"]

DIAG_LABELS = [
    "great_wall_visible_from_space",
    "bats_are_blind",
    "antibiotics_kill_viruses",
    "vikings_wore_horned_helmets",
    "seasons_caused_by_distance_from_sun"
]
diag_items = [m for m in misconceptions if m["misconception_label"] in DIAG_LABELS]
print(f"Loaded {len(diag_items)} items.")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="cuda:0", trust_remote_code=True
)
torch.cuda.empty_cache(); gc.collect()

hparams = ROMEHyperParams.from_hparams("hparams/ROME/Qwen2ForCausalLM.yaml")
hparams.model_parallel = False
hparams.device = 0
editor = BaseEditor.from_hparams(hparams)
print("Model + editor ready.")

# ── 5. Run diagnostic ────────────────────────────────────────────────────────
print("\n=== Qwen2.5-1.5B + ROME Diagnostic ===\n")
results = []
for item in diag_items:
    q = item["question"]
    opts = item["options"]
    ck = item["correct"]
    mk = item["misconception"]

    # subject extraction
    words = q.split()
    skip = {"Is","Did","Do","Does","Can","Are","Was","Were","Have","Has"}
    start = 1 if words[0] in skip else 0
    subject = None
    for length in range(min(6, len(words)-start), 0, -1):
        c = " ".join(words[start:start+length])
        if c in q: subject = c; break
    if not subject: subject = words[start]

    pre = score_mc(model, tokenizer, q, opts, ck, mk)

    # Pre-edit generation
    inp = tokenizer(q, return_tensors="pt").to(model.device)
    with torch.no_grad():
        g = model.generate(**inp, max_new_tokens=25, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id)
    pre_gen = tokenizer.decode(g[0][inp["input_ids"].shape[1]:],
                                skip_special_tokens=True).strip()[:60]

    # Edit
    _, em, _ = editor.edit(
        prompts=[q], rephrase_prompts=[q],
        target_new=[opts[ck]],
        subject=[subject], keep_original_weight=True
    )

    post = score_mc(em, tokenizer, q, opts, ck, mk)

    inp2 = tokenizer(q, return_tensors="pt").to(em.device)
    with torch.no_grad():
        g2 = em.generate(**inp2, max_new_tokens=25, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    post_gen = tokenizer.decode(g2[0][inp2["input_ids"].shape[1]:],
                                 skip_special_tokens=True).strip()[:60]

    del em; torch.cuda.empty_cache(); gc.collect()

    r = {
        "label": item["misconception_label"],
        "pre_mc": pre["predicted"], "post_mc": post["predicted"],
        "pre_misc_score": round(pre["misc_score"], 3),
        "post_misc_score": round(post["misc_score"], 3),
        "mc_changed": pre["predicted"] != post["predicted"],
        "gen_changed": pre_gen[:40] != post_gen[:40],
        "pre_gen": pre_gen, "post_gen": post_gen
    }
    results.append(r)
    print(f"[{r['label']}]")
    print(f"  MC:  {r['pre_mc']} → {r['post_mc']} | misc: {r['pre_misc_score']} → {r['post_misc_score']} | changed={r['mc_changed']}")
    print(f"  GEN: pre='{r['pre_gen']}'\n       post='{r['post_gen']}'")
    print(f"  gen_changed={r['gen_changed']}\n")

# ── 6. Summary ───────────────────────────────────────────────────────────────
print("=== SUMMARY ===")
print(f"MC changed:  {sum(r['mc_changed'] for r in results)}/5")
print(f"Gen changed: {sum(r['gen_changed'] for r in results)}/5")

06/07/2026 17:50:10 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06/07/2026 17:50:10 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json "HTTP/1.1 200 OK"


EasyEdit ready.
Hparams written.
Loaded 5 items.


06/07/2026 17:50:10 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
06/07/2026 17:50:10 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/tokenizer_config.json "HTTP/1.1 200 OK"
06/07/2026 17:50:10 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
06/07/2026 17:50:10 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
06/07/2026 17:50:11 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-1.5B-Instruct "HTTP/1.1 200 OK"
06/07/2026 17:50:11 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

06/07/2026 17:50:13 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
06/07/2026 17:50:13 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-1.5B-Instruct/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/generation_config.json "HTTP/1.1 200 OK"
06/07/2026 17:50:13 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"
2026-06-07 17:50:14,173 - easyeditor.editors.editor - INFO - Instantiating model
2026-06-07 17:50:14,173 - easyeditor.editors.editor - INFO - Instantiating model
2026-06-07 17:50:14,173 - easyeditor.editors.editor - INFO - Instantiating model
06/07/2026 17:50:14 - INFO - easyeditor.editors.editor -   Instantiating model
06/07/2026 17:50:14 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/res

...

Model + editor ready.

=== Qwen2.5-1.5B + ROME Diagnostic ===



The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  0%|          | 0/1 [00:00<?, ?it/s]

Executing ROME algorithm for the update: [Is the Great Wall of China visible from space with the naked eye?] -> [ No, it is too narrow (~9m wide) to see without optical aids]
Computing left vector (u)...
Selected u projection object the Great Wall of China visible
Left vector shape: torch.Size([8960])
Computing right vector (v)
Lookup index found: 6 | Sentence: Is the Great Wall of China visible from space with the naked eye? No, it is too narrow (~9m wide) to see without optical | Token:  visible
Rewrite layer is 5
Tying optimization objective to 27
Recording initial value of v*


  0%|          | 0/1 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 30.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 19.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 13.98 GiB is allocated by PyTorch, and 432.94 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

---
## SECTION 5: ROME — Full 48 Misconceptions (TinyLlama-1.1B)

In [ ]:
# Cell 5.0 — Install all dependencies for Sections 5-7
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
    "einops", "hydra-core", "higher", "omegaconf",
    "sentence-transformers", "peft", "sentencepiece",
    "rouge", "av", "qwen-vl-utils", "iopath",
    "fairscale", "zhipuai", "-q"], capture_output=True)
print("Dependencies installed.")


In [ ]:
import sys, os, gc, torch
import torch.nn.functional as F
import pandas as pd
sys.path.insert(0, '/kaggle/working/EasyEdit')
os.chdir('/kaggle/working/EasyEdit')

import json, yaml, shutil
from transformers import AutoTokenizer, AutoModelForCausalLM
from easyeditor import BaseEditor, ROMEHyperParams

JSON_PATH = "/kaggle/input/datasets/kevinsam77/mythbench-dataset/MythBench_v10.json"
with open(JSON_PATH) as f:
    data = json.load(f)
misconceptions = data["misconceptions"]
print(f"Total misconceptions loaded: {len(misconceptions)}")

CLUSTERS = {
    "Health_HumanBiology": [
        "antibiotics_kill_viruses","sugar_causes_hyperactivity_children",
        "tongue_taste_zones","hair_nails_grow_after_death",
        "carrots_improve_night_vision","cold_weather_causes_colds",
        "cracking_knuckles_causes_arthritis","humans_lose_most_heat_through_head",
        "humans_swallow_spiders_in_sleep","coffee_dehydrates_you",
        "left_brain_right_brain_dominance","shaving_makes_hair_grow_thicker",
        "alcohol_warms_you_up"
    ],
    "Animals_NaturalScience": [
        "bats_are_blind","camels_store_water_in_humps",
        "bulls_enraged_by_red_color","chameleons_camouflage_surroundings",
        "ostriches_bury_head_in_sand"
    ],
    "Physics_Earth_Space": [
        "great_wall_visible_from_space","seasons_caused_by_distance_from_sun",
        "toilet_flush_coriolis_hemispheres","glass_is_slow_liquid",
        "Mount_Everest_coldest_place_earth","Mt_everest_highest_point_atmosphere",
        "water_drains_opposite_hemispheres","diamond_hardest_material_earth"
    ],
    "History_Civilization": [
        "vikings_wore_horned_helmets","marie_antoinette_let_them_eat_cake",
        "columbus_proved_earth_round","cleopatra_was_egyptian",
        "nero_fiddled_while_rome_burned","pilgrims_wore_black_and_white",
        "roman_vomitoriums_for_vomiting","isaac_newton_apple_fell_on_head",
        "witch_trials_burned_at_stake_in_salem","great_wall_built_to_keep_mongols_out",
        "ben_franklin_discovered_electricity","lincoln_born_in_log_cabin_poverty",
        "cold_war_never_had_direct_combat","signing_declaration_independence_july_4",
        "walt_disney_drew_mickey_mouse"
    ],
    "Geography_CulturalOrigins": [
        "fortune_cookies_chinese_origin","french_fries_invented_in_france",
        "chinese_invented_pasta_marco_polo","america_named_after_amerigo_vespucci",
        "alaska_is_northernmost_us_state","pacific_ocean_named_for_being_calm",
        "mount_olympus_home_of_gods_in_clouds"
    ]
}
label_to_cluster = {lbl: c for c, lbls in CLUSTERS.items() for lbl in lbls}
print("Cluster assignment complete.")
for c, lbls in CLUSTERS.items():
    print(f"  {c}: {len(lbls)}")


In [ ]:
def score_mc_belief_v2(model, tokenizer, question, options, correct_key, misconception_key):
    scores = {}
    question_prompt = f"{question}\nAnswer:"
    question_ids = tokenizer.encode(question_prompt, return_tensors="pt").to(model.device)
    for key, option_text in options.items():
        full_ids = tokenizer.encode(f"{question}\nAnswer: {option_text}",
                                     return_tensors="pt").to(model.device)
        option_len = full_ids.shape[1] - question_ids.shape[1]
        if option_len <= 0:
            scores[key] = -999.0; continue
        with torch.no_grad():
            logits = model(full_ids).logits[0]
        lp = torch.nn.functional.log_softmax(logits, dim=-1)
        start = question_ids.shape[1] - 1
        scores[key] = sum(lp[start+i, full_ids[0, question_ids.shape[1]+i].item()].item()
                          for i in range(option_len)) / option_len
    best = max(scores, key=scores.get)
    return {"predicted_key": best,
            "correct_prob": scores[correct_key],
            "misconception_prob": scores[misconception_key],
            "is_correct": best == correct_key,
            "is_misconception": best == misconception_key,
            "raw_scores": scores}

def extract_subject(question):
    words = question.split()
    skip = {"Is","Did","Do","Does","Can","Are","Was","Were","Have","Has","What","Why","How","Which","Who"}
    start = 1 if words[0] in skip else 0
    subject = None
    for length in range(min(6, len(words)-start), 0, -1):
        candidate = " ".join(words[start:start+length])
        if candidate in question:
            subject = candidate
            break
    return subject if subject else words[start]

SAVE_PATH = "/kaggle/working/MisEdit_Full48_TinyLlama.csv"
MODEL_ID  = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
HPARAMS   = "hparams/ROME/LlamaForCausalLM.yaml"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="cuda:0", trust_remote_code=True)
torch.cuda.empty_cache(); gc.collect()

hparams = ROMEHyperParams.from_hparams(HPARAMS)
hparams.model_parallel = False
hparams.device = 0
editor = BaseEditor.from_hparams(hparams)
print(f"Model loaded. Running ROME on {len(misconceptions)} misconceptions...")

# Checkpoint resume
done_labels = set()
rows = []
if os.path.exists(SAVE_PATH):
    df_existing = pd.read_csv(SAVE_PATH)
    rows = df_existing.to_dict('records')
    done_labels = set(df_existing['label'].tolist())
    print(f"Resuming from checkpoint: {len(done_labels)} already done.")

for idx, item in enumerate(misconceptions):
    label = item["misconception_label"]
    if label in done_labels:
        print(f"  [{idx+1}/48] SKIP: {label}")
        continue
    question = item["question"]
    options  = item["options"]
    ck, mk   = item["correct"], item["misconception"]
    cluster  = label_to_cluster.get(label, "Unknown")
    subject  = extract_subject(question)

    pre_mc = score_mc_belief_v2(model, tokenizer, question, options, ck, mk)
    inp = tokenizer(question, return_tensors="pt").to(model.device)
    with torch.no_grad():
        g = model.generate(**inp, max_new_tokens=25, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id)
    pre_gen = tokenizer.decode(g[0][inp["input_ids"].shape[1]:],
                                skip_special_tokens=True).strip()[:80]
    try:
        metrics, edited_model, _ = editor.edit(
            prompts=[question], rephrase_prompts=[question],
            target_new=[options[ck]], subject=[subject], keep_original_weight=True)
        rw_pre  = float(metrics[0]['pre']['rewrite_acc'][0])
        rw_post = float(metrics[0]['post']['rewrite_acc'][0])
        post_mc = score_mc_belief_v2(edited_model, tokenizer, question, options, ck, mk)
        inp2 = tokenizer(question, return_tensors="pt").to(edited_model.device)
        with torch.no_grad():
            g2 = edited_model.generate(**inp2, max_new_tokens=25, do_sample=False,
                                        pad_token_id=tokenizer.eos_token_id)
        post_gen = tokenizer.decode(g2[0][inp2["input_ids"].shape[1]:],
                                     skip_special_tokens=True).strip()[:80]
        del edited_model; torch.cuda.empty_cache(); gc.collect()
        row = {
            "label": label, "cluster": cluster,
            "correct_key": ck, "misconception_key": mk,
            "rewrite_pre": round(rw_pre,4), "rewrite_post": round(rw_post,4),
            "rewrite_delta": round(rw_post-rw_pre,4),
            "pre_misc_score": round(pre_mc["misconception_prob"],4),
            "post_misc_score": round(post_mc["misconception_prob"],4),
            "mc_score_delta": round(post_mc["misconception_prob"]-pre_mc["misconception_prob"],4),
            "pre_mc_predicted": pre_mc["predicted_key"],
            "post_mc_predicted": post_mc["predicted_key"],
            "mc_changed": pre_mc["predicted_key"] != post_mc["predicted_key"],
            "pre_held_misconception": pre_mc["is_misconception"],
            "post_held_misconception": post_mc["is_misconception"],
            "gen_changed": pre_gen[:40] != post_gen[:40],
            "rewrite_improved": rw_post > rw_pre,
            "mismatch": (rw_post > rw_pre) and not (pre_mc["predicted_key"] != post_mc["predicted_key"]),
            "error": ""
        }
        status = "MISMATCH" if row["mismatch"] else "OK"
        print(f"  [{idx+1}/48] {label[:42]:<42} rw:{rw_pre:.2f}->{rw_post:.2f} mc:{row['mc_changed']} [{status}]")
    except Exception as e:
        row = {"label":label,"cluster":cluster,"correct_key":ck,"misconception_key":mk,
               "rewrite_pre":None,"rewrite_post":None,"rewrite_delta":None,
               "pre_misc_score":round(pre_mc["misconception_prob"],4),
               "post_misc_score":None,"mc_score_delta":None,
               "pre_mc_predicted":pre_mc["predicted_key"],"post_mc_predicted":None,
               "mc_changed":None,"pre_held_misconception":pre_mc["is_misconception"],
               "post_held_misconception":None,"gen_changed":None,
               "rewrite_improved":None,"mismatch":None,"error":str(e)[:200]}
        print(f"  [{idx+1}/48] ERROR: {label} — {str(e)[:60]}")
    rows.append(row)
    pd.DataFrame(rows).to_csv(SAVE_PATH, index=False)

del model; torch.cuda.empty_cache(); gc.collect()
df_full = pd.DataFrame(rows)
print(f"\nDone. {len(df_full)} items. Errors: {(df_full['error']!='').sum()}")
print(df_full[['label','rewrite_pre','rewrite_post','mc_changed','mismatch']].to_string(index=False))


---
## SECTION 6: MEMIT Attempt — 5 Items (TinyLlama-1.1B)

Try MEMIT on 5 items. If it fails or takes >30 min, skip and state as limitation.

In [ ]:
import sys, os, gc, torch
import torch.nn.functional as F
import pandas as pd
sys.path.insert(0, '/kaggle/working/EasyEdit')
os.chdir('/kaggle/working/EasyEdit')

from easyeditor import BaseEditor, MEMITHyperParams

MEMIT_HPARAMS = "hparams/MEMIT/LlamaForCausalLM.yaml"
MEMIT_5_LABELS = [
    "great_wall_visible_from_space", "bats_are_blind",
    "antibiotics_kill_viruses", "vikings_wore_horned_helmets",
    "seasons_caused_by_distance_from_sun"
]
memit_items = [m for m in misconceptions if m["misconception_label"] in MEMIT_5_LABELS]
print(f"MEMIT test items: {len(memit_items)}")

try:
    hparams_memit = MEMITHyperParams.from_hparams(MEMIT_HPARAMS)
    hparams_memit.model_parallel = False
    hparams_memit.device = 0
    editor_memit = BaseEditor.from_hparams(hparams_memit)
    print("MEMIT editor loaded successfully.")
    MEMIT_AVAILABLE = True
except Exception as e:
    print(f"MEMIT load failed: {e}")
    MEMIT_AVAILABLE = False


In [ ]:
def score_mc_belief_v2(model, tokenizer, question, options, correct_key, misconception_key):
    scores = {}
    question_prompt = f"{question}\nAnswer:"
    question_ids = tokenizer.encode(question_prompt, return_tensors="pt").to(model.device)
    for key, option_text in options.items():
        full_ids = tokenizer.encode(f"{question}\nAnswer: {option_text}",
                                     return_tensors="pt").to(model.device)
        option_len = full_ids.shape[1] - question_ids.shape[1]
        if option_len <= 0:
            scores[key] = -999.0; continue
        with torch.no_grad():
            logits = model(full_ids).logits[0]
        lp = torch.nn.functional.log_softmax(logits, dim=-1)
        start = question_ids.shape[1] - 1
        scores[key] = sum(lp[start+i, full_ids[0, question_ids.shape[1]+i].item()].item()
                          for i in range(option_len)) / option_len
    best = max(scores, key=scores.get)
    return {"predicted_key": best,
            "correct_prob": scores[correct_key],
            "misconception_prob": scores[misconception_key],
            "is_correct": best == correct_key,
            "is_misconception": best == misconception_key,
            "raw_scores": scores}

def extract_subject(question):
    words = question.split()
    skip = {"Is","Did","Do","Does","Can","Are","Was","Were","Have","Has","What","Why","How","Which","Who"}
    start = 1 if words[0] in skip else 0
    subject = None
    for length in range(min(6, len(words)-start), 0, -1):
        candidate = " ".join(words[start:start+length])
        if candidate in question:
            subject = candidate
            break
    return subject if subject else words[start]

from transformers import AutoTokenizer, AutoModelForCausalLM

SAVE_MEMIT = "/kaggle/working/MisEdit_MEMIT_5items.csv"

if not MEMIT_AVAILABLE:
    print("MEMIT not available — skipping. Will state as limitation in paper.")
else:
    MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    tokenizer_m = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer_m.pad_token is None:
        tokenizer_m.pad_token = tokenizer_m.eos_token
    model_m = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.float16, device_map="cuda:0", trust_remote_code=True)
    torch.cuda.empty_cache(); gc.collect()

    memit_rows = []
    for item in memit_items:
        label    = item["misconception_label"]
        question = item["question"]
        options  = item["options"]
        ck, mk   = item["correct"], item["misconception"]
        subject  = extract_subject(question)
        pre_mc   = score_mc_belief_v2(model_m, tokenizer_m, question, options, ck, mk)
        try:
            metrics, edited_m, _ = editor_memit.edit(
                prompts=[question], rephrase_prompts=[question],
                target_new=[options[ck]], subject=[subject], keep_original_weight=True)
            rw_pre  = float(metrics[0]['pre']['rewrite_acc'][0])
            rw_post = float(metrics[0]['post']['rewrite_acc'][0])
            post_mc = score_mc_belief_v2(edited_m, tokenizer_m, question, options, ck, mk)
            inp2 = tokenizer_m(question, return_tensors="pt").to(edited_m.device)
            with torch.no_grad():
                g2 = edited_m.generate(**inp2, max_new_tokens=25, do_sample=False,
                                        pad_token_id=tokenizer_m.eos_token_id)
            post_gen = tokenizer_m.decode(g2[0][inp2["input_ids"].shape[1]:],
                                           skip_special_tokens=True).strip()[:80]
            del edited_m; torch.cuda.empty_cache(); gc.collect()
            memit_rows.append({
                "label": label, "method": "MEMIT",
                "rewrite_pre": round(rw_pre,4), "rewrite_post": round(rw_post,4),
                "rewrite_delta": round(rw_post-rw_pre,4),
                "pre_misc_score": round(pre_mc["misconception_prob"],4),
                "post_misc_score": round(post_mc["misconception_prob"],4),
                "mc_score_delta": round(post_mc["misconception_prob"]-pre_mc["misconception_prob"],4),
                "mc_changed": pre_mc["predicted_key"] != post_mc["predicted_key"],
                "gen_changed": False, "error": ""
            })
            print(f"  {label[:42]:<42} rw:{rw_pre:.2f}->{rw_post:.2f} mc:{memit_rows[-1]['mc_changed']}")
        except Exception as e:
            print(f"  ERROR {label}: {e}")
            memit_rows.append({"label":label,"method":"MEMIT","error":str(e)[:200]})

    del model_m; torch.cuda.empty_cache(); gc.collect()
    df_memit = pd.DataFrame(memit_rows)
    df_memit.to_csv(SAVE_MEMIT, index=False)
    print(f"\nMEMIT results saved to {SAVE_MEMIT}")
    print(df_memit[['label','rewrite_pre','rewrite_post','mc_changed']].to_string(index=False))


---
## SECTION 7: Statistical Significance Analysis

In [ ]:
# Cell 7.1 — Statistical significance + full summary
import pandas as pd, json, numpy as np
from scipy import stats

df = pd.read_csv("/kaggle/working/MisEdit_Full48_TinyLlama.csv")
df_valid = df[df['error'].isna() | (df['error'] == '')].copy()
n = len(df_valid)
print(f"Valid items: {n} / {len(df)}")

rewrite_improved = int(df_valid['rewrite_improved'].sum())
mc_changed       = int(df_valid['mc_changed'].sum())
gen_changed      = int(df_valid['gen_changed'].sum())
mismatch         = int(df_valid['mismatch'].sum())
pre_held         = int(df_valid['pre_held_misconception'].sum())

print(f"\nPre-edit misconception held:   {pre_held}/{n} ({100*pre_held/n:.1f}%)")
print(f"Rewrite accuracy improved:     {rewrite_improved}/{n} ({100*rewrite_improved/n:.1f}%)")
print(f"MC belief changed:             {mc_changed}/{n} ({100*mc_changed/n:.1f}%)")
print(f"Generation changed:            {gen_changed}/{n} ({100*gen_changed/n:.1f}%)")
print(f"Rewrite-Belief Mismatch:       {mismatch}/{n} ({100*mismatch/n:.1f}%)")
print(f"\nMean rewrite delta:  {df_valid['rewrite_delta'].mean():.4f}")
print(f"Std  rewrite delta:  {df_valid['rewrite_delta'].std():.4f}")
print(f"Mean MC score delta: {df_valid['mc_score_delta'].mean():.6f}")
print(f"Std  MC score delta: {df_valid['mc_score_delta'].std():.6f}")

r1 = stats.binomtest(mc_changed, n=n, p=0.5, alternative='less')
r2 = stats.binomtest(rewrite_improved, n=n, p=0.5, alternative='greater')
print(f"\nBinomial p (MC changes, H0=0.5):      {r1.pvalue:.2e}")
print(f"Binomial p (rewrite improves, H0=0.5): {r2.pvalue:.2e}")

print("\n=== Per-Cluster ===")
cs = df_valid.groupby('cluster').agg(
    n=('label','count'), pre_held=('pre_held_misconception','sum'),
    rw_improved=('rewrite_improved','sum'), mc_changed=('mc_changed','sum'),
    mismatch=('mismatch','sum'), mean_rw_delta=('rewrite_delta','mean')
).reset_index()
print(cs.to_string(index=False))

summary = {
    "n_valid": n, "rewrite_improved_n": rewrite_improved,
    "rewrite_improved_pct": round(100*rewrite_improved/n,1),
    "mc_changed_n": mc_changed, "mc_changed_pct": round(100*mc_changed/n,1),
    "mismatch_n": mismatch, "mismatch_pct": round(100*mismatch/n,1),
    "pre_held_n": pre_held, "pre_held_pct": round(100*pre_held/n,1),
    "binomial_mc_p": round(r1.pvalue,10),
    "binomial_rw_p": round(r2.pvalue,10),
    "mean_rewrite_delta": round(float(df_valid['rewrite_delta'].mean()),4),
    "mean_mc_score_delta": round(float(df_valid['mc_score_delta'].mean()),8)
}
with open("/kaggle/working/MisEdit_Stats_Summary.json","w") as f:
    json.dump(summary, f, indent=2)
print("\nSaved: /kaggle/working/MisEdit_Stats_Summary.json")


---
## SECTION 8: Save All Outputs

In [ ]:
# Cell 8.1 — List all output files
import os
files = [
    "/kaggle/working/MisEdit_Full48_TinyLlama.csv",
    "/kaggle/working/MisEdit_MEMIT_5items.csv",
    "/kaggle/working/MisEdit_Stats_Summary.json",
]
print("=== Output Files ===")
for f in files:
    if os.path.exists(f):
        print(f"  {f}  ({os.path.getsize(f):,} bytes)  [OK]")
    else:
        print(f"  {f}  [NOT FOUND]")
print("\nDownload all three before ending session.")


In [6]:
!pip install einops hydra-core higher omegaconf sentence-transformers peft sentencepiece rouge av qwen-vl-utils iopath fairscale zhipuai statsmodels -q


In [7]:

# ============================================================
# PHASE 2 — All remaining experiments (paste as new cell)
# Runs: positive control, MC v1 vs v2, TOST+stats, 
#       error analysis, token-level probs
# Saves: 5 output files to /kaggle/working/
# ============================================================
import os, sys, json, torch, gc
import torch.nn.functional as F
import pandas as pd
import numpy as np
from scipy import stats
from transformers import AutoTokenizer, AutoModelForCausalLM

sys.path.insert(0, '/kaggle/working/EasyEdit')
os.chdir('/kaggle/working/EasyEdit')

from easyeditor import BaseEditor, ROMEHyperParams

# ── Shared helpers ────────────────────────────────────────────
def score_mc_v2(model, tokenizer, question, options, ck, mk):
    scores = {}
    q_ids = tokenizer.encode(f"{question}\nAnswer:", return_tensors="pt").to(model.device)
    for key, text in options.items():
        full_ids = tokenizer.encode(f"{question}\nAnswer: {text}",
                                    return_tensors="pt").to(model.device)
        opt_len = full_ids.shape[1] - q_ids.shape[1]
        if opt_len <= 0: scores[key] = -999.0; continue
        with torch.no_grad():
            logits = model(full_ids).logits[0]
        lp = F.log_softmax(logits, dim=-1)
        start = q_ids.shape[1] - 1
        scores[key] = sum(lp[start+i, full_ids[0, q_ids.shape[1]+i].item()].item()
                          for i in range(opt_len)) / opt_len
    best = max(scores, key=scores.get)
    return {"predicted_key": best,
            "misconception_prob": scores[mk],
            "correct_prob": scores[ck],
            "is_misconception": best == mk,
            "raw_scores": scores}

def score_mc_v1(model, tokenizer, question, options, ck, mk):
    option_text = "\n".join([f"{k}: {v}" for k,v in options.items()])
    prompt = f"{question}\n{option_text}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :]
    probs = F.softmax(logits, dim=-1)
    scores = {}
    for key in options:
        tid = tokenizer.encode(f" {key}", add_special_tokens=False)
        if not tid: tid = tokenizer.encode(key, add_special_tokens=False)
        scores[key] = probs[tid[0]].item() if tid else 0.0
    total = sum(scores.values()) + 1e-10
    norm = {k: v/total for k,v in scores.items()}
    best = max(norm, key=norm.get)
    return {"predicted_key": best, "is_misconception": best == mk,
            "misconception_prob": norm[mk], "correct_prob": norm[ck]}

def extract_subject(question):
    words = question.split()
    skip = {"Is","Did","Do","Does","Can","Are","Was","Were","Have","Has",
            "What","Why","How","Which","Who"}
    start = 1 if words[0] in skip else 0
    for length in range(min(6, len(words)-start), 0, -1):
        c = " ".join(words[start:start+length])
        if c in question: return c
    return words[start]

def load_model(model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0"):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=torch.float16, device_map="cuda:0", trust_remote_code=True)
    torch.cuda.empty_cache(); gc.collect()
    return mdl, tok

def load_editor():
    hp = ROMEHyperParams.from_hparams("hparams/ROME/LlamaForCausalLM.yaml")
    hp.model_parallel = False; hp.device = 0
    return BaseEditor.from_hparams(hp)

# Load MythBench
JSON_PATH = "/kaggle/input/datasets/kevinsam77/mythbench-dataset/MythBench_v10.json"
with open(JSON_PATH) as f: data = json.load(f)
misconceptions = data["misconceptions"]

# Load existing full48 results
df48 = pd.read_csv("/kaggle/working/MisEdit_Full48_TinyLlama.csv")
print("Setup complete.")

# ============================================================
# E2: POSITIVE CONTROL — factual edit, check MC change
# ============================================================
print("\n=== E2: Positive Control ===")

# Positive control item (not in MythBench — simple geography)
PC_ITEMS = [
    {
        "label": "eiffel_tower_location",
        "question": "In which city is the Eiffel Tower located?",
        "options": {"A": "London", "B": "Berlin", "C": "Paris", "D": "Rome"},
        "correct": "C",   # Paris — true answer
        "edit_target": "Berlin",  # edit to wrong answer
        "subject": "Eiffel Tower"
    },
    {
        "label": "water_boiling_point",
        "question": "At what temperature does water boil at sea level?",
        "options": {"A": "50 degrees Celsius", "B": "100 degrees Celsius",
                    "C": "150 degrees Celsius", "D": "200 degrees Celsius"},
        "correct": "B",
        "edit_target": "50 degrees Celsius",
        "subject": "water"
    }
]

model, tokenizer = load_model()
editor = load_editor()

pc_rows = []
for item in PC_ITEMS:
    q = item["question"]
    opts = item["options"]
    ck = item["correct"]
    # For positive control, misconception key = the edit target option
    mk = [k for k,v in opts.items() if v == item["edit_target"]][0]
    subject = item["subject"]

    pre_v2 = score_mc_v2(model, tokenizer, q, opts, ck, mk)
    pre_v1 = score_mc_v1(model, tokenizer, q, opts, ck, mk)

    try:
        metrics, em, _ = editor.edit(
            prompts=[q], rephrase_prompts=[q],
            target_new=[item["edit_target"]],
            subject=[subject], keep_original_weight=True)
        rw_pre  = float(metrics[0]['pre']['rewrite_acc'][0])
        rw_post = float(metrics[0]['post']['rewrite_acc'][0])

        post_v2 = score_mc_v2(em, tokenizer, q, opts, ck, mk)
        post_v1 = score_mc_v1(em, tokenizer, q, opts, ck, mk)
        del em; torch.cuda.empty_cache(); gc.collect()

        row = {
            "label": item["label"],
            "question": q,
            "edit_target": item["edit_target"],
            "rewrite_pre": round(rw_pre,4),
            "rewrite_post": round(rw_post,4),
            "rewrite_delta": round(rw_post-rw_pre,4),
            "rewrite_improved": rw_post > rw_pre,
            "pre_mc_v2": pre_v2["predicted_key"],
            "post_mc_v2": post_v2["predicted_key"],
            "mc_v2_changed": pre_v2["predicted_key"] != post_v2["predicted_key"],
            "pre_mc_v1": pre_v1["predicted_key"],
            "post_mc_v1": post_v1["predicted_key"],
            "mc_v1_changed": pre_v1["predicted_key"] != post_v1["predicted_key"],
            "pre_correct_score_v2": round(pre_v2["correct_prob"],4),
            "post_correct_score_v2": round(post_v2["correct_prob"],4),
            "pre_target_score_v2": round(pre_v2["misconception_prob"],4),
            "post_target_score_v2": round(post_v2["misconception_prob"],4),
            "mc_score_delta_v2": round(post_v2["misconception_prob"]-pre_v2["misconception_prob"],4),
        }
        print(f"  [{item['label']}] rw:{rw_pre:.2f}->{rw_post:.2f} mc_v2:{row['mc_v2_changed']} mc_v1:{row['mc_v1_changed']}")
    except Exception as e:
        row = {"label": item["label"], "error": str(e)[:200]}
        print(f"  ERROR: {e}")
    pc_rows.append(row)

pd.DataFrame(pc_rows).to_csv("/kaggle/working/positive_control.csv", index=False)
print("Saved: positive_control.csv")

# ============================================================
# E3: MC V1 vs V2 COMPARISON — 5 items
# ============================================================
print("\n=== E3: MC v1 vs v2 Comparison ===")

V1V2_LABELS = [
    "great_wall_visible_from_space",
    "bats_are_blind",
    "antibiotics_kill_viruses",
    "vikings_wore_horned_helmets",
    "seasons_caused_by_distance_from_sun"
]
v1v2_items = [m for m in misconceptions if m["misconception_label"] in V1V2_LABELS]

v1v2_rows = []
for item in v1v2_items:
    q = item["question"]
    opts = item["options"]
    ck, mk = item["correct"], item["misconception"]
    subject = extract_subject(q)

    pre_v2 = score_mc_v2(model, tokenizer, q, opts, ck, mk)
    pre_v1 = score_mc_v1(model, tokenizer, q, opts, ck, mk)

    try:
        metrics, em, _ = editor.edit(
            prompts=[q], rephrase_prompts=[q],
            target_new=[opts[ck]], subject=[subject], keep_original_weight=True)
        rw_pre  = float(metrics[0]['pre']['rewrite_acc'][0])
        rw_post = float(metrics[0]['post']['rewrite_acc'][0])

        post_v2 = score_mc_v2(em, tokenizer, q, opts, ck, mk)
        post_v1 = score_mc_v1(em, tokenizer, q, opts, ck, mk)
        del em; torch.cuda.empty_cache(); gc.collect()

        v1v2_rows.append({
            "label": item["misconception_label"],
            "rewrite_pre": round(rw_pre,4),
            "rewrite_post": round(rw_post,4),
            "rewrite_improved": rw_post > rw_pre,
            "v1_pre_predicted": pre_v1["predicted_key"],
            "v1_post_predicted": post_v1["predicted_key"],
            "v1_changed": pre_v1["predicted_key"] != post_v1["predicted_key"],
            "v1_pre_misc_prob": round(pre_v1["misconception_prob"],4),
            "v1_post_misc_prob": round(post_v1["misconception_prob"],4),
            "v2_pre_predicted": pre_v2["predicted_key"],
            "v2_post_predicted": post_v2["predicted_key"],
            "v2_changed": pre_v2["predicted_key"] != post_v2["predicted_key"],
            "v2_pre_misc_score": round(pre_v2["misconception_prob"],4),
            "v2_post_misc_score": round(post_v2["misconception_prob"],4),
            "v2_delta": round(post_v2["misconception_prob"]-pre_v2["misconception_prob"],6),
        })
        print(f"  {item['misconception_label'][:40]:<40} rw:{rw_pre:.2f}->{rw_post:.2f} v1_changed:{v1v2_rows[-1]['v1_changed']} v2_changed:{v1v2_rows[-1]['v2_changed']}")
    except Exception as e:
        print(f"  ERROR: {e}")

pd.DataFrame(v1v2_rows).to_csv("/kaggle/working/mc_v1_v2_comparison.csv", index=False)
print("Saved: mc_v1_v2_comparison.csv")

del model, tokenizer, editor
torch.cuda.empty_cache(); gc.collect()

# ============================================================
# E4+S1-S5: TOST + ALL STATS (no model needed)
# ============================================================
print("\n=== E4+S1-S5: Full Statistical Analysis ===")

df = df48.copy()
df_valid = df[df['error'].isna() | (df['error'] == '')].copy()
n = len(df_valid)
mc_deltas = df_valid['mc_score_delta'].values

# Basic counts
rw_improved = int(df_valid['rewrite_improved'].sum())
mc_changed   = int(df_valid['mc_changed'].sum())
pre_held     = int(df_valid['pre_held_misconception'].sum())
mismatch     = int(df_valid['mismatch'].sum())

# Binomial tests
binom_mc = stats.binomtest(mc_changed, n=n, p=0.5, alternative='less')
binom_rw = stats.binomtest(rw_improved, n=n, p=0.5, alternative='greater')

# 95% CI for zero MC-change rate (Wilson interval)
from statsmodels.stats.proportion import proportion_confint
ci_low, ci_high = proportion_confint(mc_changed, n, alpha=0.05, method='wilson')

# TOST equivalence test on MC score deltas
# H0: |mean delta| >= epsilon (epsilon = 0.01, practically meaningful change)
epsilon = 0.01
mean_delta = mc_deltas.mean()
std_delta  = mc_deltas.std(ddof=1)
se         = std_delta / np.sqrt(n)
# Two one-sided t-tests
t_upper = (epsilon - mean_delta) / se   # H0: mean >= epsilon
t_lower = (mean_delta + epsilon) / se   # H0: mean <= -epsilon
df_t    = n - 1
p_upper = stats.t.cdf(t_upper, df_t)
p_lower = stats.t.cdf(t_lower, df_t)
tost_p  = max(p_upper, p_lower)  # TOST p = max of the two

# Cohen's d for rewrite delta
rw_deltas = df_valid['rewrite_delta'].values
cohens_d  = rw_deltas.mean() / rw_deltas.std(ddof=1)

print(f"n valid: {n}")
print(f"Pre-edit misconception held: {pre_held}/{n} ({100*pre_held/n:.1f}%)")
print(f"Rewrite improved: {rw_improved}/{n} ({100*rw_improved/n:.1f}%)")
print(f"MC changed: {mc_changed}/{n}")
print(f"Mismatch: {mismatch}/{n}")
print(f"Mean rewrite delta: {rw_deltas.mean():.4f} +/- {rw_deltas.std():.4f}")
print(f"Mean MC delta: {mean_delta:.8f} +/- {std_delta:.8f}")
print(f"Binomial p (MC, H0=0.5): {binom_mc.pvalue:.2e}")
print(f"Binomial p (rw, H0=0.5): {binom_rw.pvalue:.2e}")
print(f"95% CI for MC change rate: [{ci_low:.4f}, {ci_high:.4f}]")
print(f"TOST p (epsilon=0.01): {tost_p:.6f} {'EQUIVALENT' if tost_p < 0.05 else 'NOT EQUIVALENT'}")
print(f"Cohen's d (rewrite delta): {cohens_d:.4f}")

full_stats = {
    "n_valid": n,
    "pre_held_n": pre_held, "pre_held_pct": round(100*pre_held/n,1),
    "rewrite_improved_n": rw_improved, "rewrite_improved_pct": round(100*rw_improved/n,1),
    "mc_changed_n": mc_changed, "mc_changed_pct": 0.0,
    "mismatch_n": mismatch, "mismatch_pct": round(100*mismatch/n,1),
    "mean_rewrite_delta": round(float(rw_deltas.mean()),4),
    "std_rewrite_delta": round(float(rw_deltas.std()),4),
    "mean_mc_delta": round(float(mean_delta),8),
    "std_mc_delta": round(float(std_delta),8),
    "binomial_mc_p": float(binom_mc.pvalue),
    "binomial_rw_p": round(float(binom_rw.pvalue),8),
    "ci_mc_change_low": round(float(ci_low),4),
    "ci_mc_change_high": round(float(ci_high),4),
    "tost_p_epsilon_001": round(float(tost_p),6),
    "tost_equivalent": bool(tost_p < 0.05),
    "cohens_d_rewrite": round(float(cohens_d),4),
    "tost_epsilon": epsilon
}
with open("/kaggle/working/full_stats.json","w") as f:
    json.dump(full_stats, f, indent=2)
print("Saved: full_stats.json")

# ============================================================
# E5: ERROR ANALYSIS — why 10 items failed rewrite
# ============================================================
print("\n=== E5: Error Analysis ===")

failed = df_valid[df_valid['rewrite_improved'] == False].copy()
improved = df_valid[df_valid['rewrite_improved'] == True].copy()

print(f"Failed rewrite improvement: {len(failed)}/48")
print(f"Failed items:")
for _, row in failed.iterrows():
    print(f"  {row['label'][:45]:<45} rw:{row['rewrite_pre']:.3f}->{row['rewrite_post']:.3f} delta:{row['rewrite_delta']:.3f} pre_held:{row['pre_held_misconception']}")

# Categorize failures
ea_rows = []
for _, row in failed.iterrows():
    if row['rewrite_pre'] >= 0.7:
        category = "High_pre_accuracy"
        reason = "Rewrite acc already high pre-edit, ceiling effect"
    elif row['rewrite_delta'] < -0.01:
        category = "Negative_delta"
        reason = "Edit slightly degraded rewrite accuracy"
    elif row['rewrite_delta'] == 0.0:
        category = "No_change"
        reason = "Edit had zero effect on rewrite accuracy"
    else:
        category = "Marginal_improvement"
        reason = "Small positive delta but below threshold"
    ea_rows.append({
        "label": row['label'],
        "cluster": row['cluster'],
        "rewrite_pre": row['rewrite_pre'],
        "rewrite_post": row['rewrite_post'],
        "rewrite_delta": row['rewrite_delta'],
        "pre_held_misconception": row['pre_held_misconception'],
        "category": category,
        "reason": reason
    })

df_ea = pd.DataFrame(ea_rows)
print("\nFailure categories:")
print(df_ea['category'].value_counts().to_string())
df_ea.to_csv("/kaggle/working/error_analysis.csv", index=False)
print("Saved: error_analysis.csv")

# ============================================================
# E6: TOKEN-LEVEL PROBABILITY — great_wall pre/post edit
# ============================================================
print("\n=== E6: Token-Level Probability Analysis ===")

model, tokenizer = load_model()
editor = load_editor()

gw_item = next(m for m in misconceptions
               if m["misconception_label"] == "great_wall_visible_from_space")

q    = gw_item["question"]
opts = gw_item["options"]
ck, mk = gw_item["correct"], gw_item["misconception"]

def get_option_token_probs(model, tokenizer, question, options):
    rows = []
    q_prompt = f"{question}\nAnswer:"
    q_ids = tokenizer.encode(q_prompt, return_tensors="pt").to(model.device)
    for key, text in options.items():
        full_ids = tokenizer.encode(f"{question}\nAnswer: {text}",
                                     return_tensors="pt").to(model.device)
        opt_len = full_ids.shape[1] - q_ids.shape[1]
        if opt_len <= 0: continue
        with torch.no_grad():
            logits = model(full_ids).logits[0]
        lp = F.log_softmax(logits, dim=-1)
        start = q_ids.shape[1] - 1
        tokens = tokenizer.convert_ids_to_tokens(
            full_ids[0, q_ids.shape[1]:].tolist())
        for i in range(opt_len):
            token_id = full_ids[0, q_ids.shape[1]+i].item()
            rows.append({
                "option_key": key,
                "option_text": text[:40],
                "token_pos": i,
                "token": tokens[i],
                "log_prob": round(lp[start+i, token_id].item(), 4),
                "is_correct": key == ck,
                "is_misconception": key == mk,
                "stage": "pre"
            })
    return rows

# Pre-edit
token_rows = get_option_token_probs(model, tokenizer, q, opts)

# Apply ROME
subject = extract_subject(q)
_, em, _ = editor.edit(
    prompts=[q], rephrase_prompts=[q],
    target_new=[opts[ck]], subject=[subject], keep_original_weight=True)

# Post-edit
post_rows = get_option_token_probs(em, tokenizer, q, opts)
for r in post_rows: r["stage"] = "post"
del em; torch.cuda.empty_cache(); gc.collect()

all_token_rows = token_rows + post_rows
df_tok = pd.DataFrame(all_token_rows)
df_tok.to_csv("/kaggle/working/token_probs.csv", index=False)
print(f"Token prob rows saved: {len(df_tok)}")

# Quick summary
for key in opts:
    pre_avg  = df_tok[(df_tok['option_key']==key) & (df_tok['stage']=='pre')]['log_prob'].mean()
    post_avg = df_tok[(df_tok['option_key']==key) & (df_tok['stage']=='post')]['log_prob'].mean()
    label = "[CORRECT]" if key==ck else "[MISC]" if key==mk else ""
    print(f"  Option {key} {label}: pre={pre_avg:.4f}  post={post_avg:.4f}  delta={post_avg-pre_avg:.4f}")

del model, tokenizer, editor
torch.cuda.empty_cache(); gc.collect()

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n=== ALL PHASE 2 FILES SAVED ===")
files = [
    "/kaggle/working/positive_control.csv",
    "/kaggle/working/mc_v1_v2_comparison.csv",
    "/kaggle/working/full_stats.json",
    "/kaggle/working/error_analysis.csv",
    "/kaggle/working/token_probs.csv",
]
for f in files:
    size = os.path.getsize(f) if os.path.exists(f) else 0
    status = "OK" if size > 0 else "MISSING"
    print(f"  {f.split('/')[-1]:<35} {size:>8} bytes  [{status}]")

print("\nDownload all 5 files before ending session.")


/kaggle/working/EasyEdit/easyeditor/trainer/algs/higher_utils/utils.py:15: SyntaxWarning: invalid escape sequence '\ '
  """Utility functions for components of ``higher``\ ."""
/usr/local/lib/python3.12/dist-packages/timm/models/hub.py:4: FutureWarning: Importing from timm.models.hub is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/kaggle/working/EasyEdit/easyeditor/models/melo/peft_egg/src/peft/tuners/lora.py:233: SyntaxWarning: invalid escape sequence '\.'
  layer_index = re.match(f".*.{pattern}\.(\d+)\.*", key)
/kaggle/working/EasyEdit/easyeditor/models/melo/peft_egg/src/peft/tuners/melo.py:183: SyntaxWarning: invalid escape sequence '\.'
  layer_index = re.match(f".*.{pattern}\.(\d+)\.*", key)
06/09/2026 08:34:22 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06/09/2026 08:3

Setup complete.

=== E2: Positive Control ===


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

06/09/2026 08:34:22 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
06/09/2026 08:34:22 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/tokenizer_config.json "HTTP/1.1 200 OK"
06/09/2026 08:34:22 - INFO - httpx -   HTTP Request: GET https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

06/09/2026 08:35:11 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
06/09/2026 08:35:11 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/generation_config.json "HTTP/1.1 200 OK"
06/09/2026 08:35:11 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
06/09/2026 08:35:11 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
06/09/2026 08:35:12 - INFO - httpx -   HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
06/09/2026 08:35:12

Executing ROME algorithm for the update: [Is the Great Wall of China visible from space with the naked eye?] -> [ No, it is too narrow (~9m wide) to see without optical aids]
Computing left vector (u)...
Selected u projection object the Great Wall of China visible
Left vector shape: torch.Size([5632])
Computing right vector (v)
Lookup index found: 7 | Sentence: Is the Great Wall of China visible from space with the naked eye? No, it is too narrow (~9m wide) to see without optical a | Token: visible
Rewrite layer is 5
Tying optimization objective to 21
Recording initial value of v*
loss 3.066 = 3.066 + 0.0 + 0.0 avg prob of [ No, it is too narrow (~9m wide) to see without optical aids] 0.04696408286690712
loss 2.804 = 2.722 + 0.079 + 0.004 avg prob of [ No, it is too narrow (~9m wide) to see without optical aids] 0.06608611345291138
loss 2.685 = 2.634 + 0.048 + 0.004 avg prob of [ No, it is too narrow (~9m wide) to see without optical aids] 0.0723019540309906
loss 2.391 = 2.347 + 0.04 +

2026-06-09 08:35:16,758 - easyeditor.editors.editor - INFO - 0 editing: Is the Great Wall of China visible from space with the naked eye? -> No, it is too narrow (~9m wide) to see without optical aids  

 {'pre': {'rewrite_acc': [np.float64(0.2222222222222222)], 'portability': {}, 'rephrase_acc': [np.float64(0.2222222222222222)]}, 'case_id': 0, 'requested_rewrite': {'prompt': 'Is the Great Wall of China visible from space with the naked eye?', 'target_new': 'No, it is too narrow (~9m wide) to see without optical aids', 'ground_truth': '<|endoftext|>', 'portability': {}, 'locality': {}, 'subject': 'the Great Wall of China visible', 'rephrase_prompt': 'Is the Great Wall of China visible from space with the naked eye?'}, 'post': {'rewrite_acc': [np.float64(0.2777777777777778)], 'locality': {}, 'portability': {}, 'rephrase_acc': [np.float64(0.2777777777777778)]}}
2026-06-09 08:35:16,758 - easyeditor.editors.editor - INFO - 0 editing: Is the Great Wall of China visible from space with the n

loss 2.578 = 2.537 + 0.037 + 0.004 avg prob of [ No, it is too narrow (~9m wide) to see without optical aids] 0.07930634170770645
Delta norm: 4.3984375
Change in target norm: 1.099609375 to 4.5390625 => 3.439453125
Division Factor: 1.24609375
Right vector norm: 3.529296875
Right vector shape: torch.Size([2048])
Deltas successfully computed for ['model.layers.5.mlp.down_proj.weight']
New weights successfully inserted into ['model.layers.5.mlp.down_proj.weight']
Metrics Summary:  {'pre': {'rewrite_acc': np.float64(0.2222222222222222), 'rephrase_acc': np.float64(0.2222222222222222)}, 'post': {'rewrite_acc': np.float64(0.2777777777777778), 'rephrase_acc': np.float64(0.2777777777777778)}}


Token prob rows saved: 88
  Option A : pre=-3.4806  post=-3.4806  delta=0.0000
  Option B [CORRECT]: pre=-2.9070  post=-2.9070  delta=0.0000
  Option C [MISC]: pre=-2.1654  post=-2.1654  delta=0.0000
  Option D : pre=-3.4106  post=-3.4106  delta=0.0000

=== ALL PHASE 2 FILES SAVED ===
  positive_control.csv                     572 bytes  [OK]
  mc_v1_v2_comparison.csv                  695 bytes  [OK]
  full_stats.json                          564 bytes  [OK]
  error_analysis.csv                      1368 bytes  [OK]
  token_probs.csv                         6848 bytes  [OK]

Download all 5 files before ending session.
